In [1]:
import sys, os, glob
sys.path.insert(0, os.path.abspath('../src'))
from utils.spark_session import get_spark
from pyspark.sql import functions as F

spark = get_spark()
print(spark)

26/04/07 21:12:06 WARN Utils: Your hostname, Arjuns-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 10.0.0.234 instead (on interface en0)
26/04/07 21:12:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/arjunpillai/.ivy2/cache
The jars for the packages stored in: /Users/arjunpillai/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5803dc3a-4c74-488d-9d2f-4ce22de2e851;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central


:: loading settings :: url = jar:file:/Users/arjunpillai/Library/Python/3.9/lib/python/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 86ms :: artifacts dl 5ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.1.0 from central in [default]
	io.delta#delta-storage;3.1.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0   ||   3   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-5803dc3a-4c74-488d-9d2f-4ce22de2e851
	confs: [default]
	0 artifacts copied, 3 already retrieved (0kB/3ms)
26/04/07 21:12:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-ja

In [2]:
# Paths
CMS_RAW_DIR = '../data/raw/cms/'
BRONZE_CMS = '../data/bronze/cms_cost_reports'

# Get all CMS CSV files in the raw folder
# glob.glob() returns a list of file paths matching a pattern
cms_files = glob.glob(CMS_RAW_DIR + '*.csv')
print(f'Found {len(cms_files)} CMS cost report files: {cms_files}')

# Read all files at once into a single Spark DataFrame
# Spark can read a list of files and union them automatically
df_cms_raw = (
    spark.read
    .option('header', True)
    .option('inferSchema', True)
    .option('multiLine', True)  # Handles quoted strings with newlines
    .csv(cms_files)
)

# Add a metadata column recording which file each row came from
# This is good practice for Bronze — preserves data lineage
df_cms_raw = df_cms_raw.withColumn(
    'source_file',
    F.input_file_name()  # Built-in Spark function returning the source file path
)

print(f'Total rows across all years: {df_cms_raw.count()}')
df_cms_raw.printSchema()

Found 13 CMS cost report files: ['../data/raw/cms/CostReport_2017_Final.csv', '../data/raw/cms/CostReport_2022_Final.csv', '../data/raw/cms/CostReport_2011_Final.csv', '../data/raw/cms/CostReport_2016_Final.csv', '../data/raw/cms/CostReport_2023_Final.csv', '../data/raw/cms/CostReport_2018_Final.csv', '../data/raw/cms/CostReport_2020_Final.csv', '../data/raw/cms/CostReport_2015_Final.csv', '../data/raw/cms/CostReport_2013_Final.csv', '../data/raw/cms/CostReport_2019_Final.csv', '../data/raw/cms/CostReport_2021_Final.csv', '../data/raw/cms/CostReport_2014_Final.csv', '../data/raw/cms/CostReport_2012_Final.csv']


Total rows across all years: 80077
root
 |-- rpt_rec_num: integer (nullable = true)
 |-- Provider CCN: integer (nullable = true)
 |-- Hospital Name: string (nullable = true)
 |-- Street Address: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State Code: string (nullable = true)
 |-- Zip Code: string (nullable = true)
 |-- County: string (nullable = true)
 |-- Medicare CBSA Number: integer (nullable = true)
 |-- Rural Versus Urban: string (nullable = true)
 |-- CCN Facility Type: string (nullable = true)
 |-- Provider Type: integer (nullable = true)
 |-- Type of Control: integer (nullable = true)
 |-- Fiscal Year Begin Date: string (nullable = true)
 |-- Fiscal Year End Date: string (nullable = true)
 |-- FTE - Employees on Payroll: double (nullable = true)
 |-- Number of Interns and Residents (FTE): double (nullable = true)
 |-- Total Days Title V: integer (nullable = true)
 |-- Total Days Title XVIII: integer (nullable = true)
 |-- Total Days Title XIX: integer (nul

In [3]:
import re

def sanitize_col_name(name):
    # Replace spaces and invalid Delta chars with underscores
    # Invalid chars are: space , ; { } ( ) \n \t =
    return re.sub(r'[ ,;{}()\n\t=]+', '_', name).strip('_')

# Rename all columns
sanitized_df = df_cms_raw
for col in df_cms_raw.columns:
    sanitized_df = sanitized_df.withColumnRenamed(col, sanitize_col_name(col))

print('Sanitized column names:')
for col in sanitized_df.columns:
    print(f'  {col}')

# Write to Delta
(
    sanitized_df
    .write
    .format('delta')
    .mode('overwrite')
    .save(BRONZE_CMS)
)

print(f'\nCMS Bronze layer written to {BRONZE_CMS}')

# Verify
df_check = spark.read.format('delta').load(BRONZE_CMS)
print(f'Bronze row count: {df_check.count()}')

Sanitized column names:
  rpt_rec_num
  Provider_CCN
  Hospital_Name
  Street_Address
  City
  State_Code
  Zip_Code
  County
  Medicare_CBSA_Number
  Rural_Versus_Urban
  CCN_Facility_Type
  Provider_Type
  Type_of_Control
  Fiscal_Year_Begin_Date
  Fiscal_Year_End_Date
  FTE_-_Employees_on_Payroll
  Number_of_Interns_and_Residents_FTE
  Total_Days_Title_V
  Total_Days_Title_XVIII
  Total_Days_Title_XIX
  Total_Days_V_+_XVIII_+_XIX_+_Unknown
  Number_of_Beds
  Total_Bed_Days_Available
  Total_Discharges_Title_V
  Total_Discharges_Title_XVIII
  Total_Discharges_Title_XIX
  Total_Discharges_V_+_XVIII_+_XIX_+_Unknown
  Number_of_Beds_+_Total_for_all_Subproviders
  Hospital_Total_Days_Title_V_For_Adults_&_Peds
  Hospital_Total_Days_Title_XVIII_For_Adults_&_Peds
  Hospital_Total_Days_Title_XIX_For_Adults_&_Peds
  Hospital_Total_Days_V_+_XVIII_+_XIX_+_Unknown_For_Adults_&_Peds
  Hospital_Number_of_Beds_For_Adults_&_Peds
  Hospital_Total_Bed_Days_Available_For_Adults_&_Peds
  Hospital_Total_

26/04/07 21:12:12 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.



CMS Bronze layer written to ../data/bronze/cms_cost_reports
Bronze row count: 80077


In [4]:
USDA_RAW = '../data/raw/usda/rucc_2013.csv'
BRONZE_USDA = '../data/bronze/usda_rucc'

df_rucc = (
    spark.read
    .option('header', True)
    .option('inferSchema', True)
    .csv(USDA_RAW)
)

print(f'RUCC rows: {df_rucc.count()}')
df_rucc.show(5)

df_rucc.write.format('delta').mode('overwrite').save(BRONZE_USDA)
print(f'USDA Bronze layer written to {BRONZE_USDA}')

RUCC rows: 3234
+----+-----+--------------+---------------+---------+--------------------+
|FIPS|State|   County_Name|Population_2010|RUCC_2013|         Description|
+----+-----+--------------+---------------+---------+--------------------+
|1001|   AL|Autauga County|          54571|      2.0|Metro - Counties ...|
|1003|   AL|Baldwin County|         182265|      3.0|Metro - Counties ...|
|1005|   AL|Barbour County|          27457|      6.0|Nonmetro - Urban ...|
|1007|   AL|   Bibb County|          22915|      1.0|Metro - Counties ...|
|1009|   AL| Blount County|          57322|      1.0|Metro - Counties ...|
+----+-----+--------------+---------------+---------+--------------------+
only showing top 5 rows

USDA Bronze layer written to ../data/bronze/usda_rucc


In [5]:
SHEPS_RAW = '../data/raw/sheps/rural_closures.csv'
BRONZE_SHEPS = '../data/bronze/sheps_closures'

df_closures = (
    spark.read
    .option('header', True)
    .option('inferSchema', True)
    .csv(SHEPS_RAW)
)

print(f'Closure records: {df_closures.count()}')
df_closures.show(10)

df_closures.write.format('delta').mode('overwrite').save(BRONZE_SHEPS)
print(f'Sheps Bronze layer written to {BRONZE_SHEPS}')

Closure records: 110
+--------------------+------------+-----+------------+
|       hospital_name|        city|state|closure_year|
+--------------------+------------+-----+------------+
|Stilwell Memorial...|    STILWELL|   OK|        2025|
|Northern Light In...|  WATERVILLE|   ME|        2025|
|Mid Coast Medical...|     TRINITY|   TX|        2025|
|Valley Community ...|PAULS VALLEY|   OK|        2025|
|MercyOne Primghar...|    PRIMGHAR|   IA|        2024|
|Thomasville Regio...| THOMASVILLE|   AL|        2024|
|Community Memoria...|  HICKSVILLE|   OH|        2024|
|Jellico Regional ...|     JELLICO|   TN|        2024|
|St. Mark's Medica...|   LA GRANGE|   TX|        2023|
|  Herington Hospital|   HERINGTON|   KS|        2023|
+--------------------+------------+-----+------------+
only showing top 10 rows

Sheps Bronze layer written to ../data/bronze/sheps_closures


26/04/08 04:44:49 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 852488 ms exceeds timeout 120000 ms
26/04/08 04:44:49 WARN SparkContext: Killing executors is not supported by current scheduler.
26/04/08 04:59:59 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$